# SupportOps AI - RAG Retrieval Pipeline

## Objective

Build the retrieval layer for the SupportOps AI knowledge assistant.

### Pipeline

Support Policies
→ Document Loading
→ Chunking
→ Embeddings
→ Vector Database
→ Semantic Search
→ Relevant Policy Sections

The initial implementation uses Sentence Transformers and ChromaDB.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
import chromadb

print("RAG libraries imported successfully!")

In [ ]:
KNOWLEDGE_BASE_DIR = Path(
    "../knowledge_base"
)

documents = []

for file_path in sorted(
    KNOWLEDGE_BASE_DIR.glob("*.md")
):
    
    text = file_path.read_text(
        encoding="utf-8"
    )

    documents.append({
        "source": file_path.name,
        "content": text
    })

print(
    "Documents loaded:",
    len(documents)
)

In [ ]:
for document in documents:
    print("=" * 70)
    print("SOURCE:", document["source"])
    print(document["content"][:500])

Create a simple Markdown chunker

In [ ]:
def chunk_markdown_document(
    text,
    source
):
    
    chunks = []

    sections = text.split("\n## ")

    for index, section in enumerate(sections):

        section = section.strip()

        if not section:
            continue

        if index == 0:
            chunk_text = section
        else:
            chunk_text = "## " + section

        chunks.append({
            "source": source,
            "chunk_id": f"{source}_{index}",
            "text": chunk_text
        })

    return chunks

In [ ]:
all_chunks = []

for document in documents:

    document_chunks = (
        chunk_markdown_document(
            document["content"],
            document["source"]
        )
    )

    all_chunks.extend(
        document_chunks
    )

print(
    "Total chunks:",
    len(all_chunks)
)

In [ ]:
chunks_df = pd.DataFrame(
    all_chunks
)

chunks_df[
    ["source", "chunk_id", "text"]
]

Load the embedding model

In [ ]:
EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print(
    "Embedding model loaded successfully!"
)

Generate one embedding

In [ ]:
sample_sentence = (
    "I was charged twice for the same card payment."
)

sample_embedding = embedding_model.encode(
    sample_sentence
)

print(
    "Embedding shape:",
    sample_embedding.shape
)

print(
    sample_embedding[:10]
)

Compare semantic similarity

In [ ]:
sentences = [
    "I was charged twice for the same card payment.",
    "There is a duplicate transaction on my card.",
    "I forgot my PIN.",
    "My transfer is still pending."
]

embeddings = embedding_model.encode(
    sentences,
    normalize_embeddings=True
)

similarity_matrix = (
    embeddings @ embeddings.T
)

pd.DataFrame(
    similarity_matrix,
    index=sentences,
    columns=sentences
)

Create a persistent Chroma client

In [ ]:
CHROMA_PATH = "../rag/chroma_db"

chroma_client = chromadb.PersistentClient(
    path=CHROMA_PATH
)

print("ChromaDB client created!")

Create a collection

In [ ]:
COLLECTION_NAME = "novabank_support_policies"

try:
    chroma_client.delete_collection(
        name=COLLECTION_NAME
    )
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={
        "hnsw:space": "cosine"
    }
)

print(
    "Collection created:",
    COLLECTION_NAME
)

Generate embeddings for all chunks

In [ ]:
chunk_texts = [
    chunk["text"]
    for chunk in all_chunks
]

In [ ]:
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True
)

print(
    "Chunk embeddings shape:",
    chunk_embeddings.shape
)

Prepare metadata

In [ ]:
chunk_ids = [
    chunk["chunk_id"]
    for chunk in all_chunks
]

chunk_metadata = [
    {
        "source": chunk["source"],
        "chunk_id": chunk["chunk_id"]
    }
    for chunk in all_chunks
]

Add everything to ChromaDB

In [ ]:
collection.add(
    ids=chunk_ids,
    documents=chunk_texts,
    embeddings=chunk_embeddings.tolist(),
    metadatas=chunk_metadata
)

print(
    "Chunks stored in ChromaDB:",
    collection.count()
)

Run your first semantic search

In [ ]:
query = (
    "I was charged twice for the same card purchase."
)

In [ ]:
#Generate its embedding
query_embedding = embedding_model.encode(
    query,
    normalize_embeddings=True
)

In [ ]:
#then query chroma
results = collection.query(
    query_embeddings=[
        query_embedding.tolist()
    ],
    n_results=3
)

results

Make the results readable

In [ ]:
for i in range(
    len(results["documents"][0])
):
    
    print("=" * 80)

    print(
        "RANK:",
        i + 1
    )

    print(
        "SOURCE:",
        results["metadatas"][0][i]["source"]
    )

    print(
        "DISTANCE:",
        round(
            results["distances"][0][i],
            4
        )
    )

    print("\nCONTENT:")
    print(
        results["documents"][0][i]
    )

    print()

Create a clean retrieval function

In [ ]:
def retrieve_context(
    query,
    top_k=3
):
    
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    results = collection.query(
        query_embeddings=[
            query_embedding.tolist()
        ],
        n_results=top_k
    )

    retrieved_chunks = []

    for i in range(
        len(results["documents"][0])
    ):
        
        distance = (
            results["distances"][0][i]
        )

        retrieved_chunks.append({
            "rank": i + 1,

            "source":
                results["metadatas"][0][i][
                    "source"
                ],

            "chunk_id":
                results["metadatas"][0][i][
                    "chunk_id"
                ],

            "text":
                results["documents"][0][i],

            "distance":
                round(float(distance), 4),

            "similarity":
                round(
                    1 - float(distance),
                    4
                )
        })

    return retrieved_chunks

In [ ]:
retrieve_context(
    "I was charged twice for one card transaction."
)

Turn retrieval results into a DataFrame

In [ ]:
retrieval_results = retrieve_context(
    "I was charged twice for one card transaction."
)

pd.DataFrame(
    retrieval_results
)[
    [
        "rank",
        "source",
        "chunk_id",
        "similarity"
    ]
]

Test different business questions

In [ ]:
#duplicate card payment
retrieve_context(
    "I was charged twice for the same transaction."
)

In [ ]:
#missing refund
retrieve_context(
    "The merchant refunded me but the money still has not appeared."
)

In [ ]:
#pending transfer
retrieve_context(
    "My bank transfer has been pending for three days."
)

In [ ]:
#unknown card payment
retrieve_context(
    "There is a card transaction on my account that I do not recognise."
)

Create a retrieval test table

In [ ]:
retrieval_tests = [
    {
        "question":
            "I was charged twice for the same transaction.",
        "expected_source":
            "01_card_payments.md"
    },
    {
        "question":
            "The merchant refunded me but the money has not arrived.",
        "expected_source":
            "02_refunds.md"
    },
    {
        "question":
            "My bank transfer has been pending for three days.",
        "expected_source":
            "03_bank_transfers.md"
    },
    {
        "question":
            "I don't recognise a payment made with my card.",
        "expected_source":
            "01_card_payments.md"
    }
]

In [ ]:
evaluation_rows = []

for test in retrieval_tests:

    retrieved = retrieve_context(
        test["question"],
        top_k=3
    )

    retrieved_sources = [
        item["source"]
        for item in retrieved
    ]

    hit = (
        test["expected_source"]
        in retrieved_sources
    )

    evaluation_rows.append({
        "question":
            test["question"],

        "expected_source":
            test["expected_source"],

        "top_1_source":
            retrieved_sources[0],

        "retrieved_sources":
            retrieved_sources,

        "hit_at_3":
            hit
    })

retrieval_evaluation_df = pd.DataFrame(
    evaluation_rows
)

retrieval_evaluation_df

Calculate Hit@3

In [ ]:
hit_at_3 = (
    retrieval_evaluation_df[
        "hit_at_3"
    ].mean()
)

print(
    f"Retrieval Hit@3: "
    f"{hit_at_3:.2%}"
)

Create a context builder

In [ ]:
def build_rag_context(
    query,
    top_k=3
):
    
    retrieved = retrieve_context(
        query,
        top_k=top_k
    )

    context_blocks = []

    for item in retrieved:

        block = (
            f"[Source: {item['source']}]\n"
            f"{item['text']}"
        )

        context_blocks.append(block)

    context = "\n\n---\n\n".join(
        context_blocks
    )

    return context, retrieved

In [ ]:
context, sources = build_rag_context(
    "What should we do when a customer is charged twice?"
)

print(context)

Build the full RAG prompt

In [ ]:
def build_rag_prompt(
    question,
    context
):
    
    prompt = f"""
You are SupportOps AI, an internal customer-support assistant for NovaBank.

Your job is to help support agents resolve customer issues using ONLY the
information provided in the retrieved NovaBank policy context.

Rules:
1. Do not invent policies, timelines, fees, procedures, or requirements.
2. Base your answer only on the retrieved context.
3. If the context does not contain enough information, clearly say:
   "The available policy context does not provide enough information."
4. Give clear, actionable steps for the support agent.
5. Mention relevant timelines when they are explicitly stated in the context.
6. Cite the source document used for each important recommendation.
7. Do not claim that an action has already been completed.
8. Keep the response concise and professional.

RETRIEVED POLICY CONTEXT
------------------------
{context}

CUSTOMER QUESTION
-----------------
{question}

SUPPORT RECOMMENDATION
----------------------
"""
    
    return prompt.strip()

In [ ]:
question = (
    "I was charged twice for the same card transaction. "
    "What should I do?"
)

context, retrieved_sources = build_rag_context(
    question,
    top_k=3
)

prompt = build_rag_prompt(
    question,
    context
)

print(prompt)

Add source labels more cleanly

In [ ]:
def build_rag_context(
    query,
    top_k=3
):
    
    retrieved = retrieve_context(
        query,
        top_k=top_k
    )

    context_blocks = []

    for index, item in enumerate(
        retrieved,
        start=1
    ):
        
        block = (
            f"[Source {index}: "
            f"{item['source']} | "
            f"{item['chunk_id']}]\n"
            f"{item['text']}"
        )

        context_blocks.append(block)

    context = "\n\n---\n\n".join(
        context_blocks
    )

    return context, retrieved

Add source labels more cleanly

In [ ]:
def build_rag_context(
    query,
    top_k=3
):
    
    retrieved = retrieve_context(
        query,
        top_k=top_k
    )

    context_blocks = []

    for index, item in enumerate(
        retrieved,
        start=1
    ):
        
        block = (
            f"[Source {index}: "
            f"{item['source']} | "
            f"{item['chunk_id']}]\n"
            f"{item['text']}"
        )

        context_blocks.append(block)

    context = "\n\n---\n\n".join(
        context_blocks
    )

    return context, retrieved

In [ ]:
context, retrieved = build_rag_context(
    "I was charged twice for the same transaction."
)

print(context)

Add a retrieval confidence guard

In [ ]:
def retrieve_context(
    query,
    top_k=3,
    min_similarity=None
):
    
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    results = collection.query(
        query_embeddings=[
            query_embedding.tolist()
        ],
        n_results=top_k
    )

    retrieved_chunks = []

    for i in range(
        len(results["documents"][0])
    ):
        
        distance = float(
            results["distances"][0][i]
        )

        similarity = 1 - distance

        if (
            min_similarity is not None
            and similarity < min_similarity
        ):
            continue

        retrieved_chunks.append({
            "rank": i + 1,
            "source":
                results["metadatas"][0][i]["source"],
            "chunk_id":
                results["metadatas"][0][i]["chunk_id"],
            "text":
                results["documents"][0][i],
            "distance":
                round(distance, 4),
            "similarity":
                round(similarity, 4)
        })

    return retrieved_chunks

Test an out-of-domain question

In [ ]:
out_of_domain_results = retrieve_context(
    "What is the weather in London tomorrow?",
    top_k=3
)

pd.DataFrame(
    out_of_domain_results
)[
    [
        "rank",
        "source",
        "chunk_id",
        "similarity"
    ]
]

Test an out-of-domain question

In [ ]:
out_of_domain_results = retrieve_context(
    "What is the weather in London tomorrow?",
    top_k=3
)

pd.DataFrame(
    out_of_domain_results
)[
    [
        "rank",
        "source",
        "chunk_id",
        "similarity"
    ]
]

Reload all 8 documents

In [ ]:
KNOWLEDGE_BASE_DIR = Path(
    "../knowledge_base"
)

documents = []

for file_path in sorted(
    KNOWLEDGE_BASE_DIR.glob("*.md")
):

    text = file_path.read_text(
        encoding="utf-8"
    )

    documents.append({
        "source": file_path.name,
        "content": text
    })

print(
    "Documents loaded:",
    len(documents)
)

In [ ]:
all_chunks = []

for document in documents:

    document_chunks = (
        chunk_markdown_document(
            document["content"],
            document["source"]
        )
    )

    all_chunks.extend(
        document_chunks
    )

print(
    "Total chunks:",
    len(all_chunks)
)

Inspect the expanded knowledge base

In [ ]:
chunks_df = pd.DataFrame(
    all_chunks
)

print(
    chunks_df.groupby("source")
    .size()
)

Rebuild the embeddings

In [ ]:
chunk_texts = [
    chunk["text"]
    for chunk in all_chunks
]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True
)

print(
    "Chunk embeddings shape:",
    chunk_embeddings.shape
)

Rebuild ChromaDB

In [ ]:
COLLECTION_NAME = (
    "novabank_support_policies"
)

try:
    chroma_client.delete_collection(
        name=COLLECTION_NAME
    )
except Exception:
    pass

collection = (
    chroma_client.create_collection(
        name=COLLECTION_NAME,
        metadata={
            "hnsw:space": "cosine"
        }
    )
)

In [ ]:
chunk_ids = [
    chunk["chunk_id"]
    for chunk in all_chunks
]

chunk_metadata = [
    {
        "source":
            chunk["source"],

        "chunk_id":
            chunk["chunk_id"]
    }
    for chunk in all_chunks
]

In [ ]:
collection.add(
    ids=chunk_ids,
    documents=chunk_texts,
    embeddings=chunk_embeddings.tolist(),
    metadatas=chunk_metadata
)

print(
    "Chunks stored in ChromaDB:",
    collection.count()
)

Test the new topics

In [ ]:
test_questions = [
    "The ATM charged my account but did not give me any cash.",
    "My PIN is blocked after I entered it incorrectly.",
    "I lost my card and I can see a transaction I do not recognise.",
    "My identity verification keeps failing.",
    "When should a support ticket be escalated?"
]

for question in test_questions:

    print("\n" + "=" * 90)
    print("QUESTION:")
    print(question)

    results = retrieve_context(
        question,
        top_k=3
    )

    for item in results:

        print(
            f"\nRank {item['rank']} | "
            f"{item['source']} | "
            f"{item['chunk_id']} | "
            f"Similarity: "
            f"{item['similarity']}"
        )

Expand the retrieval evaluation

In [ ]:
retrieval_tests = [
    {
        "question":
            "I was charged twice for the same transaction.",
        "expected_source":
            "01_card_payments.md"
    },

    {
        "question":
            "The merchant refunded me but the money has not arrived.",
        "expected_source":
            "02_refunds.md"
    },

    {
        "question":
            "My bank transfer has been pending for three days.",
        "expected_source":
            "03_bank_transfers.md"
    },

    {
        "question":
            "The ATM charged me but did not give me any cash.",
        "expected_source":
            "04_cash_withdrawals.md"
    },

    {
        "question":
            "My PIN has been blocked after too many attempts.",
        "expected_source":
            "05_cards_and_pin.md"
    },

    {
        "question":
            "I lost my card and think somebody may be using it.",
        "expected_source":
            "06_account_security.md"
    },

    {
        "question":
            "My identity verification keeps failing.",
        "expected_source":
            "07_identity_verification.md"
    },

    {
        "question":
            "When should a customer support case be escalated?",
        "expected_source":
            "08_support_escalation.md"
    }
]

In [ ]:
evaluation_rows = []

for test in retrieval_tests:

    retrieved = retrieve_context(
        test["question"],
        top_k=3
    )

    retrieved_sources = [
        item["source"]
        for item in retrieved
    ]

    evaluation_rows.append({
        "question":
            test["question"],

        "expected_source":
            test["expected_source"],

        "top_1_source":
            retrieved_sources[0],

        "hit_at_1":
            retrieved_sources[0]
            == test["expected_source"],

        "hit_at_3":
            test["expected_source"]
            in retrieved_sources
    })

retrieval_evaluation_df = pd.DataFrame(
    evaluation_rows
)

retrieval_evaluation_df

In [ ]:
hit_at_1 = (
    retrieval_evaluation_df[
        "hit_at_1"
    ].mean()
)

hit_at_3 = (
    retrieval_evaluation_df[
        "hit_at_3"
    ].mean()
)

print(
    f"Retrieval Hit@1: "
    f"{hit_at_1:.2%}"
)

print(
    f"Retrieval Hit@3: "
    f"{hit_at_3:.2%}"
)

Saving the retrieval evaluation

In [ ]:
from pathlib import Path

RAG_EVAL_DIR = Path("../rag/evaluation")
RAG_EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

retrieval_evaluation_df.to_csv(
    RAG_EVAL_DIR / "retrieval_evaluation.csv",
    index=False
)

print("Retrieval evaluation saved.")

In [ ]:
retrieval_evaluation_df[
    retrieval_evaluation_df["hit_at_1"] == False
]

Load the key safely

In [ ]:
import os

from dotenv import load_dotenv
from google import genai

In [ ]:
load_dotenv("../.env")

gemini_api_key = os.getenv(
    "GEMINI_API_KEY"
)

if not gemini_api_key:
    raise ValueError(
        "GEMINI_API_KEY was not found in the .env file."
    )

print("Gemini API key loaded successfully!")

Create the Gemini client

In [ ]:
client = genai.Client(
    api_key=gemini_api_key
)

print("Gemini client created successfully!")

Test Gemini before touching RAG

In [ ]:
LLM_MODEL = "gemini-3.7-flash"

In [ ]:
import os

from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../.env")

gemini_api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(
    api_key=gemini_api_key,
    http_options=types.HttpOptions(
        timeout=30000
    )
)

print("Gemini client recreated with timeout.")

In [ ]:
model_info = client.models.get(
    model="gemini-3.7-flash"
)

print(model_info.name)

In [ ]:
test_response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Reply only with: Gemini connection successful",
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_level="minimal"
        ),
        max_output_tokens=20
    )
)

print(test_response.text)

In [ ]:
LLM_MODEL = "gemini-3.5-flash-lite"

print("LLM model:", LLM_MODEL)

Build the grounded RAG prompt

In [ ]:
def build_rag_prompt(
    question,
    context
):

    prompt = f"""
You are SupportOps AI, an internal customer-support assistant for NovaBank.

Your task is to answer the support question using ONLY the NovaBank policy
information provided in the retrieved context.

GROUNDING RULES

1. Use only information contained in the retrieved policy context.
2. Do not invent policies, timelines, fees, requirements, or procedures.
3. If the context does not contain enough information to answer the question,
   clearly say:
   "The available policy context does not provide enough information."
4. Give clear and actionable steps for the support agent.
5. Mention timelines only when they are explicitly stated in the policy.
6. Cite supporting policy evidence using [Source 1], [Source 2], etc.
7. Do not cite a source unless it supports the statement.
8. Do not claim that an investigation, refund, escalation, or other action
   has already been completed.
9. Keep the answer concise and professional.

RETRIEVED NOVABANK POLICY CONTEXT
---------------------------------
{context}

SUPPORT QUESTION
----------------
{question}

GROUNDED SUPPORT RECOMMENDATION
-------------------------------
"""

    return prompt.strip()

Make sure your context builder has source labels

In [ ]:
def build_rag_context(
    query,
    top_k=3
):

    retrieved = retrieve_context(
        query,
        top_k=top_k
    )

    context_blocks = []

    for index, item in enumerate(
        retrieved,
        start=1
    ):

        block = (
            f"[Source {index}: "
            f"{item['source']} | "
            f"{item['chunk_id']}]\n"
            f"{item['text']}"
        )

        context_blocks.append(
            block
        )

    context = "\n\n---\n\n".join(
        context_blocks
    )

    return context, retrieved

Create the Gemini generation function

In [ ]:
def generate_grounded_answer(
    question,
    context
):

    prompt = build_rag_prompt(
        question,
        context
    )

    response = client.models.generate_content(
        model=LLM_MODEL,
        contents=prompt
    )

    return response.text

Build the complete RAG function

In [ ]:
def answer_with_rag(
    question,
    top_k=3
):

    # STEP 1
    # Retrieve relevant NovaBank policy chunks
    context, retrieved = build_rag_context(
        question,
        top_k=top_k
    )

    # STEP 2
    # Send retrieved context + question to Gemini
    answer = generate_grounded_answer(
        question,
        context
    )

    # STEP 3
    # Return both answer and evidence
    return {
        "question": question,
        "answer": answer,
        "sources": retrieved
    }

Test 1: Duplicate card payment

In [ ]:
rag_result = answer_with_rag(
    "I was charged twice for the same card transaction. "
    "What should the support agent do?"
)

In [ ]:
print(
    rag_result["answer"]
)

Display answer + retrieval together

In [ ]:
print("QUESTION")
print("=" * 80)

print(
    rag_result["question"]
)


print("\nANSWER")
print("=" * 80)

print(
    rag_result["answer"]
)


print("\nRETRIEVED SOURCES")
print("=" * 80)

for source in rag_result["sources"]:

    print(
        f"Rank {source['rank']} | "
        f"{source['source']} | "
        f"{source['chunk_id']} | "
        f"Similarity: "
        f"{source['similarity']}"
    )

Test 2: Pending bank transfer

In [ ]:
transfer_result = answer_with_rag(
    "A customer's bank transfer has been pending "
    "for three business days. "
    "What should the support agent do?"
)

print(
    transfer_result["answer"]
)

In [ ]:
for source in transfer_result["sources"]:

    print(
        source["rank"],
        source["source"],
        source["chunk_id"],
        source["similarity"]
    )

Test 3: ATM cash problem

In [ ]:
atm_result = answer_with_rag(
    "The ATM charged the customer's account "
    "but did not dispense any cash. "
    "What should the agent do?"
)

print(
    atm_result["answer"]
)

Test hallucination resistance

In [ ]:
unsupported_result = answer_with_rag(
    "What interest rate does NovaBank pay "
    "on savings accounts?"
)

print(
    unsupported_result["answer"]
)

Completely unrelated question

In [ ]:
weather_result = answer_with_rag(
    "What is the weather going to be tomorrow?"
)

print(
    weather_result["answer"]
)

Inspect the unsupported retrieval

In [ ]:
for source in weather_result["sources"]:

    print(
        f"Rank {source['rank']} | "
        f"{source['source']} | "
        f"Similarity: "
        f"{source['similarity']}"
    )

Save our retrieval evaluation

In [ ]:
from pathlib import Path

RAG_EVAL_DIR = Path(
    "../rag/evaluation"
)

RAG_EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
retrieval_evaluation_df.to_csv(
    RAG_EVAL_DIR
    / "retrieval_evaluation.csv",
    index=False
)

print(
    "Retrieval evaluation saved!"
)

Create in-domain questions

In [ ]:
in_domain_questions = [
    "I was charged twice for the same card payment.",
    "My transfer has been pending for three days.",
    "The ATM charged me but gave me no cash.",
    "My PIN is blocked.",
    "I lost my card and someone may have used it.",
    "My identity verification keeps failing.",
    "The merchant refunded me but the money has not arrived.",
    "When should this support case be escalated?"
]

Create out-of-domain questions

In [ ]:
out_of_domain_questions = [
    "What is the weather tomorrow?",
    "Who won the football match yesterday?",
    "What is the capital of France?",
    "How do I cook pasta?",
    "What interest rate does NovaBank offer on savings accounts?",
    "Can you recommend a good laptop?",
    "What is the stock price of Apple?",
    "Write me a birthday message."
]

Measure top retrieval similarity

In [ ]:
def get_top_similarity(question):
    
    results = retrieve_context(
        question,
        top_k=1
    )
    
    if not results:
        return 0.0
    
    return results[0]["similarity"]

In [ ]:
print(
    get_top_similarity(
        "My PIN is blocked."
    )
)

Compare in-domain vs out-of-domain

In [ ]:
confidence_rows = []

for question in in_domain_questions:
    
    confidence_rows.append({
        "question": question,
        "category": "in_domain",
        "top_similarity":
            get_top_similarity(question)
    })


for question in out_of_domain_questions:
    
    confidence_rows.append({
        "question": question,
        "category": "out_of_domain",
        "top_similarity":
            get_top_similarity(question)
    })


confidence_df = pd.DataFrame(
    confidence_rows
)

confidence_df

Summarize the distributions

In [ ]:
confidence_summary = (
    confidence_df
    .groupby("category")[
        "top_similarity"
    ]
    .agg([
        "min",
        "mean",
        "max"
    ])
)

confidence_summary

Inspect the hardest cases

In [ ]:
print("LOWEST IN-DOMAIN SCORES")
print("=" * 70)

display(
    confidence_df[
        confidence_df["category"]
        == "in_domain"
    ]
    .sort_values(
        "top_similarity"
    )
    .head(5)
)


print("\nHIGHEST OUT-OF-DOMAIN SCORES")
print("=" * 70)

display(
    confidence_df[
        confidence_df["category"]
        == "out_of_domain"
    ]
    .sort_values(
        "top_similarity",
        ascending=False
    )
    .head(5)
)

Save this experiment

In [ ]:
confidence_df.to_csv(
    "../rag/evaluation/"
    "retrieval_confidence_analysis.csv",
    index=False
)

print(
    "Confidence analysis saved!"
)

Update the question categories

In [ ]:
supported_questions = [
    "I was charged twice for the same card payment.",
    "My transfer has been pending for three days.",
    "The ATM charged me but gave me no cash.",
    "My PIN is blocked.",
    "I lost my card and someone may have used it.",
    "My identity verification keeps failing.",
    "The merchant refunded me but the money has not arrived.",
    "When should this support case be escalated?"
]

unsupported_in_domain_questions = [
    "What interest rate does NovaBank offer on savings accounts?"
]

out_of_domain_questions = [
    "What is the weather tomorrow?",
    "Who won the football match yesterday?",
    "What is the capital of France?",
    "How do I cook pasta?",
    "Can you recommend a good laptop?",
    "What is the stock price of Apple?",
    "Write me a birthday message."
]

Add threshold guardrail

In [ ]:
RETRIEVAL_THRESHOLD = 0.45

In [ ]:
def format_retrieved_context(
    retrieved
):

    context_blocks = []

    for index, item in enumerate(
        retrieved,
        start=1
    ):

        block = (
            f"[Source {index}: "
            f"{item['source']} | "
            f"{item['chunk_id']}]\n"
            f"{item['text']}"
        )

        context_blocks.append(block)

    return "\n\n---\n\n".join(
        context_blocks
    )

In [ ]:
def answer_with_rag(
    question,
    top_k=3,
    threshold=RETRIEVAL_THRESHOLD
):

    # STEP 1
    # Retrieve relevant policy chunks
    retrieved = retrieve_context(
        question,
        top_k=top_k
    )

    # Handle empty retrieval
    if not retrieved:

        return {
            "question": question,
            "answer": (
                "The available policy context does not "
                "provide enough information."
            ),
            "sources": [],
            "top_similarity": 0.0,
            "status": "no_retrieval"
        }

    # STEP 2
    # Inspect strongest retrieval result
    top_similarity = (
        retrieved[0]["similarity"]
    )

    # STEP 3
    # Reject clearly unrelated questions
    if top_similarity < threshold:

        return {
            "question": question,
            "answer": (
                "The available policy context does not "
                "provide enough information."
            ),
            "sources": retrieved,
            "top_similarity":
                top_similarity,
            "status":
                "low_retrieval_confidence"
        }

    # STEP 4
    # Build context from retrieved evidence
    context = format_retrieved_context(
        retrieved
    )

    # STEP 5
    # Generate grounded response with Gemini
    answer = generate_grounded_answer(
        question,
        context
    )

    return {
        "question": question,
        "answer": answer,
        "sources": retrieved,
        "top_similarity":
            top_similarity,
        "status":
            "generated"
    }

Test the guardrail

In [ ]:
result = answer_with_rag(
    "My PIN is blocked."
)

print("Status:", result["status"])
print(
    "Similarity:",
    result["top_similarity"]
)
print(
    "Answer:",
    result["answer"]
)

Weather question

In [ ]:
result = answer_with_rag(
    "What is the weather tomorrow?"
)

print("Status:", result["status"])
print(
    "Similarity:",
    result["top_similarity"]
)
print(
    "Answer:",
    result["answer"]
)

Savings-interest question

In [ ]:
result = answer_with_rag(
    "What interest rate does NovaBank "
    "offer on savings accounts?"
)

print("Status:", result["status"])
print(
    "Similarity:",
    result["top_similarity"]
)
print(
    "Answer:",
    result["answer"]
)

Larger RAG evaluation

In [ ]:
rag_evaluation_questions = [
    # --------------------------------------------------
    # CARD PAYMENTS
    # --------------------------------------------------
    {
        "question":
            "Why was my card payment declined even though I have enough money?",
        "expected_source":
            "01_card_payments.md",
        "category":
            "supported"
    },
    {
        "question":
            "I can see the same card purchase twice on my account.",
        "expected_source":
            "01_card_payments.md",
        "category":
            "supported"
    },
    {
        "question":
            "There is a card transaction I do not recognise.",
        "expected_source":
            "01_card_payments.md",
        "category":
            "supported"
    },

    # --------------------------------------------------
    # REFUNDS
    # --------------------------------------------------
    {
        "question":
            "The merchant says they refunded me but I still cannot see it.",
        "expected_source":
            "02_refunds.md",
        "category":
            "supported"
    },
    {
        "question":
            "How long should I wait for a merchant refund to appear?",
        "expected_source":
            "02_refunds.md",
        "category":
            "supported"
    },
    {
        "question":
            "The merchant processed my refund more than ten business days ago.",
        "expected_source":
            "02_refunds.md",
        "category":
            "supported"
    },

    # --------------------------------------------------
    # TRANSFERS
    # --------------------------------------------------
    {
        "question":
            "My transfer has been pending for three business days.",
        "expected_source":
            "03_bank_transfers.md",
        "category":
            "supported"
    },
    {
        "question":
            "My bank transfer keeps failing.",
        "expected_source":
            "03_bank_transfers.md",
        "category":
            "supported"
    },
    {
        "question":
            "The transfer says completed but the recipient has not received it.",
        "expected_source":
            "03_bank_transfers.md",
        "category":
            "supported"
    },

    # --------------------------------------------------
    # CASH WITHDRAWALS
    # --------------------------------------------------
    {
        "question":
            "The ATM charged my account but did not give me cash.",
        "expected_source":
            "04_cash_withdrawals.md",
        "category":
            "supported"
    },
    {
        "question":
            "The ATM gave me less cash than the amount shown on my account.",
        "expected_source":
            "04_cash_withdrawals.md",
        "category":
            "supported"
    },
    {
        "question":
            "My cash withdrawal keeps getting declined.",
        "expected_source":
            "04_cash_withdrawals.md",
        "category":
            "supported"
    },

    # --------------------------------------------------
    # CARDS AND PIN
    # --------------------------------------------------
    {
        "question":
            "My PIN is blocked after several incorrect attempts.",
        "expected_source":
            "05_cards_and_pin.md",
        "category":
            "supported"
    },
    {
        "question":
            "My new card will not activate.",
        "expected_source":
            "05_cards_and_pin.md",
        "category":
            "supported"
    },
    {
        "question":
            "My physical card is damaged and no longer works.",
        "expected_source":
            "05_cards_and_pin.md",
        "category":
            "supported"
    },

    # --------------------------------------------------
    # SECURITY
    # --------------------------------------------------
    {
        "question":
            "I lost my card and I think someone has used it.",
        "expected_source":
            "06_account_security.md",
        "category":
            "supported"
    },
    {
        "question":
            "I think someone has gained access to my account.",
        "expected_source":
            "06_account_security.md",
        "category":
            "supported"
    },
    {
        "question":
            "I lost the phone I use to access NovaBank.",
        "expected_source":
            "06_account_security.md",
        "category":
            "supported"
    },

    # --------------------------------------------------
    # IDENTITY VERIFICATION
    # --------------------------------------------------
    {
        "question":
            "My identity verification keeps failing.",
        "expected_source":
            "07_identity_verification.md",
        "category":
            "supported"
    },
    {
        "question":
            "My account name does not match my identity document.",
        "expected_source":
            "07_identity_verification.md",
        "category":
            "supported"
    },
    {
        "question":
            "What happens when automated verification repeatedly fails?",
        "expected_source":
            "07_identity_verification.md",
        "category":
            "supported"
    },

    # --------------------------------------------------
    # ESCALATION
    # --------------------------------------------------
    {
        "question":
            "When should a support case be escalated?",
        "expected_source":
            "08_support_escalation.md",
        "category":
            "supported"
    },
    {
        "question":
            "What information should an escalation contain?",
        "expected_source":
            "08_support_escalation.md",
        "category":
            "supported"
    },
    {
        "question":
            "How should suspected account takeover be escalated?",
        "expected_source":
            "08_support_escalation.md",
        "category":
            "supported"
    },

    # --------------------------------------------------
    # UNSUPPORTED BUT BANKING RELATED
    # --------------------------------------------------
    {
        "question":
            "What interest rate does NovaBank pay on savings accounts?",
        "expected_source":
            None,
        "category":
            "unsupported_in_domain"
    },
    {
        "question":
            "Does NovaBank offer mortgages?",
        "expected_source":
            None,
        "category":
            "unsupported_in_domain"
    },
    {
        "question":
            "What is NovaBank's overdraft interest rate?",
        "expected_source":
            None,
        "category":
            "unsupported_in_domain"
    },
    {
        "question":
            "Can I open a business banking account?",
        "expected_source":
            None,
        "category":
            "unsupported_in_domain"
    },

    # --------------------------------------------------
    # TRUE OUT OF DOMAIN
    # --------------------------------------------------
    {
        "question":
            "What is the weather tomorrow?",
        "expected_source":
            None,
        "category":
            "out_of_domain"
    },
    {
        "question":
            "Who won yesterday's football match?",
        "expected_source":
            None,
        "category":
            "out_of_domain"
    },
    {
        "question":
            "Can you recommend a laptop?",
        "expected_source":
            None,
        "category":
            "out_of_domain"
    },
    {
        "question":
            "Write a birthday message for my friend.",
        "expected_source":
            None,
        "category":
            "out_of_domain"
    }
]

print(
    "Evaluation questions:",
    len(rag_evaluation_questions)
)

Evaluate retrieval without Gemini

In [ ]:
retrieval_eval_rows = []

for item in rag_evaluation_questions:

    retrieved = retrieve_context(
        item["question"],
        top_k=3
    )

    top_similarity = (
        retrieved[0]["similarity"]
        if retrieved
        else 0.0
    )

    retrieved_sources = [
        result["source"]
        for result in retrieved
    ]

    if item["category"] == "supported":

        hit_at_1 = (
            retrieved_sources[0]
            == item["expected_source"]
        )

        hit_at_3 = (
            item["expected_source"]
            in retrieved_sources
        )

    else:
        hit_at_1 = None
        hit_at_3 = None

    retrieval_eval_rows.append({
        "question":
            item["question"],

        "category":
            item["category"],

        "expected_source":
            item["expected_source"],

        "top_source":
            retrieved_sources[0]
            if retrieved_sources
            else None,

        "top_similarity":
            top_similarity,

        "hit_at_1":
            hit_at_1,

        "hit_at_3":
            hit_at_3,

        "passes_threshold":
            top_similarity
            >= RETRIEVAL_THRESHOLD
    })


rag_eval_df = pd.DataFrame(
    retrieval_eval_rows
)

rag_eval_df

Calculate retrieval metrics

In [ ]:
supported_eval = rag_eval_df[
    rag_eval_df["category"]
    == "supported"
].copy()

In [ ]:
retrieval_hit_at_1 = (
    supported_eval[
        "hit_at_1"
    ].mean()
)

retrieval_hit_at_3 = (
    supported_eval[
        "hit_at_3"
    ].mean()
)

print(
    f"Retrieval Hit@1: "
    f"{retrieval_hit_at_1:.2%}"
)

print(
    f"Retrieval Hit@3: "
    f"{retrieval_hit_at_3:.2%}"
)

Test whether threshold rejects valid questions

In [ ]:
supported_pass_rate = (
    supported_eval[
        "passes_threshold"
    ].mean()
)

print(
    f"Supported-query pass rate: "
    f"{supported_pass_rate:.2%}"
)

Evaluate true out-of-domain rejection

In [ ]:
ood_eval = rag_eval_df[
    rag_eval_df["category"]
    == "out_of_domain"
].copy()

ood_rejection_rate = (
    ~ood_eval[
        "passes_threshold"
    ]
).mean()

print(
    f"Out-of-domain rejection rate: "
    f"{ood_rejection_rate:.2%}"
)

Inspect unsupported banking queries

In [ ]:
unsupported_eval = rag_eval_df[
    rag_eval_df["category"]
    == "unsupported_in_domain"
].copy()

unsupported_eval[
    [
        "question",
        "top_source",
        "top_similarity",
        "passes_threshold"
    ]
]

Save larger retrieval evaluation

In [ ]:
rag_eval_df.to_csv(
    "../rag/evaluation/"
    "rag_retrieval_evaluation_32_queries.csv",
    index=False
)

print(
    "32-query RAG retrieval evaluation saved!"
)

Citation validation

In [ ]:
import re


def has_source_citation(answer):

    pattern = r"\[Source\s+\d+\]"

    return bool(
        re.search(
            pattern,
            answer
        )
    )

In [ ]:
print(
    has_source_citation(
        "Escalate the issue [Source 1]."
    )
)

print(
    has_source_citation(
        "Escalate the issue."
    )
)

Check whether citations are valid

In [ ]:
def extract_source_numbers(answer):

    matches = re.findall(
        r"\[Source\s+(\d+)\]",
        answer
    )

    return [
        int(number)
        for number in matches
    ]

In [ ]:
def citations_are_valid(
    answer,
    number_of_sources
):

    citation_numbers = (
        extract_source_numbers(
            answer
        )
    )

    if not citation_numbers:
        return False

    return all(
        1 <= citation
        <= number_of_sources

        for citation
        in citation_numbers
    )

In [ ]:
print(
    citations_are_valid(
        "Follow the policy [Source 1].",
        3
    )
)

In [ ]:
print(
    citations_are_valid(
        "Follow the policy [Source 7].",
        3
    )
)

Small generation-quality evaluation

In [ ]:
generation_test_questions = [
    # Supported
    "I was charged twice for the same card payment.",
    "My transfer has been pending for three business days.",
    "The ATM charged my account but did not give me cash.",
    "My PIN is blocked.",
    "I lost my card and think someone may have used it.",
    "My identity verification keeps failing.",
    "What information should an escalation contain?",

    # Unsupported in-domain
    "What interest rate does NovaBank pay on savings accounts?",
    "Does NovaBank offer mortgages?",

    # Out of domain
    "What is the weather tomorrow?",
    "Can you recommend a laptop?",
    "Write me a birthday message."
]

In [ ]:
generation_eval_rows = []

for question in generation_test_questions:

    result = answer_with_rag(
        question
    )

    generation_eval_rows.append({
        "question":
            question,

        "status":
            result["status"],

        "top_similarity":
            result["top_similarity"],

        "answer":
            result["answer"],

        "has_citation":
            has_source_citation(
                result["answer"]
            ),

        "valid_citations":
            (
                citations_are_valid(
                    result["answer"],
                    len(result["sources"])
                )
                if result["status"]
                == "generated"
                else True
            )
    })


generation_eval_df = pd.DataFrame(
    generation_eval_rows
)

generation_eval_df

Save generation evaluation

In [ ]:
generation_eval_df.to_csv(
    "../rag/evaluation/"
    "rag_generation_evaluation.csv",
    index=False
)

print(
    "Generation evaluation saved!"
)

Define the fallback message once

In [ ]:
FALLBACK_MESSAGE = (
    "The available policy context does not "
    "provide enough information."
)

In [ ]:
def is_insufficient_context_answer(answer):
    
    return FALLBACK_MESSAGE.lower() in answer.lower()

In [ ]:
print(
    is_insufficient_context_answer(
        "The available policy context does not provide enough information."
    )
)

Improve answer_with_rag()

In [ ]:
def answer_with_rag(
    question,
    top_k=3,
    threshold=RETRIEVAL_THRESHOLD
):

    # --------------------------------------------------
    # STEP 1: Retrieve relevant policy chunks
    # --------------------------------------------------
    retrieved = retrieve_context(
        question,
        top_k=top_k
    )

    if not retrieved:
        return {
            "question": question,
            "answer": FALLBACK_MESSAGE,
            "sources": [],
            "top_similarity": 0.0,
            "status": "no_retrieval"
        }

    # --------------------------------------------------
    # STEP 2: Inspect strongest retrieval result
    # --------------------------------------------------
    top_similarity = retrieved[0]["similarity"]

    # --------------------------------------------------
    # STEP 3: Reject clearly unrelated questions
    # --------------------------------------------------
    if top_similarity < threshold:
        return {
            "question": question,
            "answer": FALLBACK_MESSAGE,
            "sources": retrieved,
            "top_similarity": top_similarity,
            "status": "low_retrieval_confidence"
        }

    # --------------------------------------------------
    # STEP 4: Build retrieved context
    # --------------------------------------------------
    context = format_retrieved_context(
        retrieved
    )

    # --------------------------------------------------
    # STEP 5: Generate grounded response
    # --------------------------------------------------
    answer = generate_grounded_answer(
        question,
        context
    )

    # --------------------------------------------------
    # STEP 6: Determine whether the knowledge base
    # actually supported an answer
    # --------------------------------------------------
    if is_insufficient_context_answer(answer):
        status = "insufficient_context"
    else:
        status = "answered"

    return {
        "question": question,
        "answer": answer,
        "sources": retrieved,
        "top_similarity": top_similarity,
        "status": status
    }

Fix citation evaluation

In [ ]:
def evaluate_citation_behavior(
    result
):

    status = result["status"]
    answer = result["answer"]

    has_citation = (
        has_source_citation(answer)
    )

    if status == "answered":

        valid_citations = (
            citations_are_valid(
                answer,
                len(result["sources"])
            )
        )

        citation_behavior_correct = (
            has_citation
            and valid_citations
        )

    else:

        # Refusals should not be forced
        # to invent policy citations.
        valid_citations = None

        citation_behavior_correct = (
            not has_citation
        )

    return {
        "has_citation":
            has_citation,

        "valid_citations":
            valid_citations,

        "citation_behavior_correct":
            citation_behavior_correct
    }

Rerun generation evaluation

In [ ]:
generation_test_questions = [
    # Supported
    "I was charged twice for the same card payment.",
    "My transfer has been pending for three business days.",
    "The ATM charged my account but did not give me cash.",
    "My PIN is blocked.",
    "I lost my card and think someone may have used it.",
    "My identity verification keeps failing.",
    "What information should an escalation contain?",

    # Unsupported but banking-related
    "What interest rate does NovaBank pay on savings accounts?",
    "Does NovaBank offer mortgages?",
    "What is NovaBank's overdraft interest rate?",
    "Can I open a business banking account?",

    # Out of domain
    "What is the weather tomorrow?",
    "Can you recommend a laptop?",
    "Write me a birthday message."
]

In [ ]:
generation_eval_rows = []

for question in generation_test_questions:

    result = answer_with_rag(
        question
    )

    citation_metrics = (
        evaluate_citation_behavior(
            result
        )
    )

    generation_eval_rows.append({
        "question":
            question,

        "status":
            result["status"],

        "top_similarity":
            result["top_similarity"],

        "answer":
            result["answer"],

        **citation_metrics
    })


generation_eval_df = pd.DataFrame(
    generation_eval_rows
)

generation_eval_df

Calculate final generation metrics

In [ ]:
answered_df = generation_eval_df[
    generation_eval_df["status"]
    == "answered"
]

In [ ]:
citation_compliance_rate = (
    answered_df[
        "citation_behavior_correct"
    ].mean()
)

print(
    f"Citation compliance: "
    f"{citation_compliance_rate:.2%}"
)

In [ ]:
safe_behavior_rate = (
    generation_eval_df[
        "citation_behavior_correct"
    ].mean()
)

print(
    f"Overall citation/refusal behavior: "
    f"{safe_behavior_rate:.2%}"
)

Save final evaluation

In [ ]:
generation_eval_df.to_csv(
    "../rag/evaluation/rag_generation_evaluation.csv",
    index=False
)

print("Final generation evaluation saved!")